# Week 6: Large Language Models (LLMs)

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week06-llms.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Imports that some of the lesson's code fragments assume
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Using with Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-4-Scout-17B-16E-Instruct")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Create prompt (the tokenizer's chat template applies Llama 4's own special tokens)
messages = [
    {"role": "system", "content": "You are an expert assistant that helps users with information about technology."},
    {"role": "user", "content": "What are the key features of LLAMA4?"},
]

# Tokenize prompt
input_ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

# Generate response
outputs = model.generate(
    input_ids,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

# Decode and print response (only the newly generated tokens)
print(tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True))

## Using with vLLM

In [ ]:
from vllm import LLM, SamplingParams

# Initialize the model
llm = LLM(
    model="meta-llama/Llama-4-Maverick-17B-128E-Instruct",
    dtype="bfloat16",
    gpu_memory_utilization=0.9,
)

# Create sampling parameters
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512,
)

# Create prompt (vLLM applies the model's chat template)
messages = [
    {"role": "system", "content": "You are an expert assistant that helps users with information about technology."},
    {"role": "user", "content": "What are the key features of LLAMA4?"},
]

# Generate response
outputs = llm.chat(messages, sampling_params)

# Print response
print(outputs[0].outputs[0].text)

## Using LLAMA4 with Images

In [ ]:
from transformers import AutoProcessor, Llama4ForConditionalGeneration
import torch

# Load model and processor (requires transformers >= 4.51)
model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load image and create prompt (the processor downloads the image from the URL)
url = "https://example.com/image.jpg"
messages = [
    {"role": "user", "content": [
        {"type": "image", "url": url},
        {"type": "text", "text": "What can you see in this image?"},
    ]}
]

# Process inputs
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device)

# Generate response
outputs = model.generate(**inputs, max_new_tokens=100)

# Decode and print response (only the newly generated tokens)
print(processor.batch_decode(outputs[:, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)[0])

## Run in your browser

**Temperature, top-k and top-p sampling (NumPy)**

In [ ]:
import numpy as np

words = ["sunny", "cold", "nice", "rainy", "perfect", "Tuesday", "banana"]
logits = np.array([3.2, 2.6, 2.4, 2.1, 1.5, -0.8, -2.5])
rng = np.random.default_rng(0)

def next_token_probs(logits, temperature=1.0, top_k=None, top_p=None):
    z = logits / temperature
    p = np.exp(z - z.max()); p /= p.sum()
    keep = np.ones_like(p, dtype=bool)
    order = np.argsort(p)[::-1]
    if top_k is not None:
        keep[order[top_k:]] = False
    if top_p is not None:
        cum = np.cumsum(p[order])
        cutoff = np.searchsorted(cum, top_p) + 1          # smallest set with mass >= top_p
        keep[order[cutoff:]] = False
    p = np.where(keep, p, 0); return p / p.sum()

for T in (0.2, 1.0, 2.0):
    p = next_token_probs(logits, T)
    print(f"T={T}: " + ", ".join(f"{w}={pi:.2f}" for w, pi in zip(words, p)))

p = next_token_probs(logits, 1.0, top_p=0.9)
print("\ntop-p=0.9 keeps:", [w for w, pi in zip(words, p) if pi > 0])
print("10 samples:", [str(w) for w in rng.choice(words, size=10, p=p)])

## Test code for the course environment

**Active-parameter math for MoE models (no GPU needed)**

In [ ]:
# 1. Active-parameter math for MoE models (no GPU needed)
models = {
    "Scout":    dict(total=109e9, experts=16,  active=17e9),
    "Maverick": dict(total=400e9, experts=128, active=17e9),
    "Behemoth": dict(total=2e12,  experts=16,  active=288e9),
}
for name, m in models.items():
    frac = m["active"] / m["total"]
    print(f"{name:9s} active/total = {frac:6.1%}  "
          f"(~{m['total']*2/1e9:.0f} GB in BF16, ~{m['total']*0.5/1e9:.0f} GB in int4)")

**Softmax sampling with temperature and top-p (nucleus), as used in the generate() calls**

In [ ]:
# 2. Softmax sampling with temperature and top-p (nucleus), as used in the generate() calls
import numpy as np

def sample_next(logits, temperature=0.7, top_p=0.9, rng=np.random.default_rng(0)):
    z = np.asarray(logits) / temperature
    p = np.exp(z - z.max()); p /= p.sum()
    order = np.argsort(-p)
    cum = np.cumsum(p[order])
    keep = order[: np.searchsorted(cum, top_p) + 1]
    q = p[keep] / p[keep].sum()
    return rng.choice(keep, p=q), dict(zip(keep.tolist(), q.round(3).tolist()))

tok, kept = sample_next([2.0, 1.5, 1.0, 0.2, -1.0])
print("sampled token id:", tok, "| nucleus:", kept)

**Context-window scale: how many tokens is a document? (needs: pip install tiktoken)**

In [ ]:
# 3. Context-window scale: how many tokens is a document? (needs: pip install tiktoken)
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
text = "Large language models read text as tokens, not as words or characters. " * 200  # or: open("any_file.txt").read()
n = len(enc.encode(text))
for model, ctx in [("Llama 3", 128_000), ("Maverick", 1_000_000), ("Scout", 10_000_000)]:
    print(f"{model:9s} window holds ~{ctx // n:,} copies of this text ({n} tokens)")

**Smoke test the Hugging Face loading pattern on a tiny model (CPU-friendly stand-in for Lla**

In [ ]:
# 4. Smoke test the Hugging Face loading pattern on a tiny model (CPU-friendly stand-in for Llama 4)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
name = "sshleifer/tiny-gpt2"   # swap for "meta-llama/Llama-4-Scout-17B-16E-Instruct" on an H100
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)
inputs = tok("What are the key features of LLAMA4?", return_tensors="pt")
out = model.generate(inputs.input_ids, max_new_tokens=20, do_sample=True, temperature=0.7, top_p=0.9)
print(tok.decode(out[0], skip_special_tokens=True))